In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="10-personalized-news-feed",
    notebook_name="03_multi_task_and_engagement.ipynb"
)

# Multi-Task Learning and Engagement Scoring: Deep Dive

## The Big Idea (For a 12-Year-Old)

Imagine a school with one really smart principal who teaches ALL students the basics: how to read, how to do math, how to think clearly. THEN, there are specialist teachers: one who only teaches art, one who only teaches music, one who only teaches robotics.

The principal's lessons help EVERY specialist teacher because the students already know how to think. A student who is great at reading will pick up music theory faster. A student who is good at math will learn robotics circuits more quickly.

**Multi-task learning works the same way:**
- The **shared backbone** is the principal -- it learns patterns useful for ALL tasks
- The **task-specific heads** are the specialists -- each predicts one type of reaction
- Training to predict likes ALSO helps predict shares, because both require understanding "what do users find valuable?"

---

## Staff-Level Technical Summary

This notebook covers:
- **Why multi-task beats N independent models** (sparse data, shared representations, cost)
- **Full multi-task DNN in PyTorch**: shared backbone + 6 heads (click, like, comment, share, dwell-time, skip)
- **Training dataset construction**: positive/negative sampling, class imbalance handling
- **Loss function**: weighted sum of BCE (classification) + Huber (regression)
- **User-author affinity features**: the single most predictive signal per Facebook research
- **Weighted engagement score**: computing and visualizing the final ranking score
- **Positional bias**: explanation + inverse propensity weighting correction
- **Cold-start for new users**: content-based fallback strategy

## 1. Why Multi-Task Learning?

### The Problem with N Independent Models

Suppose we train a separate model for each reaction type (click, like, comment, share, skip).

| Problem | Explanation |
|---------|-------------|
| **Sparse reactions** | Shares happen <3% of the time. A dedicated share model starves for data |
| **Expensive** | 6 models to train, serve, and maintain instead of 1 |
| **No knowledge transfer** | The click model cannot help the share model, even though both need to understand "what is engaging?" |
| **Inconsistent representations** | Each model learns a different internal representation of users and posts |

### How Multi-Task Solves Each Problem

```
    SINGLE MODEL, N HEADS
    ====================================

    Input: (user_features, post_features, affinity_features)
                        |
                        v
    +--------------------------------------------+
    |          SHARED BACKBONE                   |
    |  Layer 1: 256 units (learns general patterns|
    |  Layer 2: 128 units (learns user-post match)|
    |  Layer 3:  64 units (refines representation)|
    +---+-----+------+------+--------+------------+
        |     |      |      |        |
        v     v      v      v        v
    +------++------++-----++------++------+
    |Click ||Like  ||Cmt  ||Share ||Dwell |
    | Head || Head ||Head || Head || Head |
    +------++------++-----++------++------+
        |     |      |      |        |
        v     v      v      v        v
    P(click) P(like) P(cmt) P(share) E[dwell_time]
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

np.random.seed(42)
torch.manual_seed(42)

# === Why Multi-Task: Visualizing the Data Sparsity Problem ===

# Simulate a realistic interaction log at scale
n_impressions = 100_000  # 100k impressions

reaction_rates = {
    'click':    0.24,   # 24% -- most common (low bar: just tap to expand)
    'like':     0.09,   # 9%  -- deliberate positive signal
    'comment':  0.04,   # 4%  -- requires effort to write something
    'share':    0.02,   # 2%  -- very deliberate, rare
    'skip':     0.38,   # 38% -- implicit negative (scrolled past in <0.5s)
    'dwell':    1.00,   # 100% -- everyone has some dwell time (even if tiny)
}

reaction_counts = {r: int(n_impressions * rate) for r, rate in reaction_rates.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Absolute counts
names = ['click', 'like', 'comment', 'share']
counts = [reaction_counts[n] for n in names]
colors = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22']
bars = axes[0].bar(names, counts, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel('Number of Events', fontsize=12)
axes[0].set_title(f'Reaction Counts (from {n_impressions:,} impressions)', fontsize=13, fontweight='bold')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Share problem (if training only share model)
training_sizes = np.logspace(3, 6, 100)
share_samples = training_sizes * reaction_rates['share']  # 2% of impressions
click_samples = training_sizes * reaction_rates['click']  # 24% of impressions

# AUC improves with log(data) -- simplified model
auc_share_independent = 0.5 + 0.15 * np.log10(np.maximum(share_samples, 1)) / 4
auc_share_multitask   = 0.5 + 0.15 * np.log10(np.maximum(click_samples, 1)) / 4  # benefits from click data
auc_share_multitask   = np.minimum(auc_share_multitask, 0.92)
auc_share_independent = np.minimum(auc_share_independent, 0.82)

axes[1].semilogx(training_sizes, auc_share_independent, 'r-', linewidth=2.5, label='Independent share model')
axes[1].semilogx(training_sizes, auc_share_multitask, 'g-', linewidth=2.5, label='Multi-task share head')
axes[1].fill_between(training_sizes, auc_share_independent, auc_share_multitask,
                     alpha=0.15, color='green', label='Multi-task advantage')
axes[1].set_xlabel('Total Impressions in Training Set', fontsize=12)
axes[1].set_ylabel('Share Prediction AUC', fontsize=12)
axes[1].set_title('Multi-Task Advantage for Sparse Tasks', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')

plt.tight_layout()
plt.show()

print("=== Why Multi-Task Wins ===")
print(f"\nFrom {n_impressions:,} impressions:")
for name in names:
    print(f"  {name:<10}: {reaction_counts[name]:>6,} samples ({reaction_rates[name]*100:.0f}%)")
print(f"\nThe share model would get only {reaction_counts['share']:,} positive training examples.")
print("With multi-task learning, it benefits from the {reaction_counts['click']:,} click examples")
print("because the shared backbone learns 'what makes content engaging' from all tasks.")

## 2. Multi-Task DNN Architecture in PyTorch

### Architecture Decision: How Many Heads?

We need one head for each reaction we care about:

| Head | Type | Output | Why |
|------|------|--------|-----|
| **click** | Binary classification | P(click) ∈ [0,1] | Most common signal, easy to collect |
| **like** | Binary classification | P(like) ∈ [0,1] | Strong explicit positive signal |
| **comment** | Binary classification | P(comment) ∈ [0,1] | User invested effort to write |
| **share** | Binary classification | P(share) ∈ [0,1] | Strongest virality signal |
| **dwell_time** | Regression | E[seconds] ≥ 0 | Captures passive engagement (lurkers!) |
| **skip** | Binary classification | P(skip) ∈ [0,1] | User scrolled past in under 0.5 seconds |

In [ ]:
# === Multi-Task DNN: Full Implementation ===

class MultiTaskNewsFeedDNN(nn.Module):
    """
    Multi-task DNN for news feed engagement prediction.
    
    12-year-old version:
    One shared 'brain' that learns what makes content engaging for a user,
    plus 6 separate 'experts' each predicting one reaction type.
    The brain's knowledge flows to ALL experts -- that's the key advantage.
    
    Staff-level detail:
    - Shared backbone: 3 layers of Linear -> BatchNorm -> ReLU -> Dropout
      BatchNorm helps with training stability (gradients stay healthy)
      Dropout prevents overfitting on the majority class
    - Binary classification heads: sigmoid activation -> probability
    - Regression head (dwell_time): ReLU activation -> non-negative seconds
    - Skip head: binary classification (1 = user scrolled past in <0.5s)
    """
    
    def __init__(self, input_dim,
                 shared_dims=(256, 128, 64),
                 head_dims=(32, 16)):
        super().__init__()
        
        self.binary_tasks   = ['click', 'like', 'comment', 'share', 'skip']
        self.regression_tasks = ['dwell_time']
        self.all_tasks      = self.binary_tasks + self.regression_tasks
        
        # ---- Shared Backbone ----
        layers = []
        prev_dim = input_dim
        for dim in shared_dims:
            layers += [
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(p=0.3),
            ]
            prev_dim = dim
        self.shared_backbone = nn.Sequential(*layers)
        self.shared_output_dim = shared_dims[-1]
        
        # ---- Task-Specific Heads ----
        self.task_heads = nn.ModuleDict()
        for task in self.all_tasks:
            head = []
            d = self.shared_output_dim
            for h_dim in head_dims:
                head += [nn.Linear(d, h_dim), nn.ReLU(), nn.Dropout(0.1)]
                d = h_dim
            head.append(nn.Linear(d, 1))
            self.task_heads[task] = nn.Sequential(*head)
    
    def forward(self, x):
        shared = self.shared_backbone(x)  # (batch, 64)
        
        out = {}
        for task in self.binary_tasks:
            out[task] = torch.sigmoid(self.task_heads[task](shared)).squeeze(-1)
        for task in self.regression_tasks:
            # ReLU ensures non-negative dwell time
            out[task] = torch.relu(self.task_heads[task](shared)).squeeze(-1)
        return out


# --- Model summary ---
INPUT_DIM = 18  # (affinity features + user features + post features -- detailed below)
model = MultiTaskNewsFeedDNN(input_dim=INPUT_DIM)

total_params = sum(p.numel() for p in model.parameters())
shared_params = sum(p.numel() for p in model.shared_backbone.parameters())
head_params = total_params - shared_params

print("=== Multi-Task DNN Architecture ===")
print(model)
print(f"\n--- Parameter Breakdown ---")
print(f"Shared backbone:  {shared_params:>8,} params ({shared_params/total_params*100:.1f}%)")
print(f"All task heads:   {head_params:>8,} params ({head_params/total_params*100:.1f}%)")
print(f"Total:            {total_params:>8,} params")
print(f"\nFor comparison: {len(model.all_tasks)} independent models would need ~{total_params * len(model.all_tasks) // 2:,} params")
print(f"Multi-task saves ~{(1 - total_params / (total_params * len(model.all_tasks) // 2)) * 100:.0f}% of parameters")

## 3. User-Author Affinity Features

### Why Affinity Is the Most Predictive Signal

According to Facebook's published engineering research, **the relationship between a user and the author of a post is the single strongest predictor of engagement**.

Intuitively this makes complete sense. Imagine two identical posts -- one posted by your best friend, one posted by a stranger you followed last year. You are far more likely to stop and read your best friend's post, like it, and comment on it.

This is called **user-author affinity** -- how much does this specific user historically engage with content from this specific author?

### The Affinity Feature Table

| Feature | How to Compute | Why It Matters |
|---------|---------------|----------------|
| **like_rate** | user's likes on author's posts / author's posts user saw | Direct revealed preference |
| **click_rate** | user's clicks on author's posts / author's posts user saw | Weaker but more frequent |
| **comment_rate** | user's comments on author's posts / author's posts user saw | High-effort positive signal |
| **hide_rate** | user's hides of author's posts / author's posts user saw | Negative affinity signal |
| **friendship_length_days** | days since they became friends | Long friendship = strong tie |
| **close_friend_flag** | did user mark author as a close friend? | Strongest possible signal |

In [ ]:
# === Computing User-Author Affinity Features ===

from datetime import datetime, timedelta

np.random.seed(42)

# Simulate interaction history
n_users, n_authors = 50, 30

# True underlying affinity (what we are trying to capture)
true_affinity = np.random.beta(0.5, 3.0, (n_users, n_authors))  # mostly low, a few high
# Sprinkle in some strong affinities (best friends)
for _ in range(20):
    u, a = np.random.randint(0, n_users), np.random.randint(0, n_authors)
    true_affinity[u, a] = np.random.uniform(0.65, 0.95)

def generate_interaction_log(n_impressions=8000):
    rows = []
    base_date = datetime(2024, 1, 1)
    for _ in range(n_impressions):
        user_id   = np.random.randint(0, n_users)
        author_id = np.random.randint(0, n_authors)
        aff       = true_affinity[user_id, author_id]
        ts = base_date + timedelta(days=np.random.uniform(0, 90))
        rows.append({'user_id': user_id, 'author_id': author_id, 'type': 'impression', 'ts': ts})
        if np.random.random() < aff * 0.6:  rows.append({'user_id': user_id, 'author_id': author_id, 'type': 'click', 'ts': ts})
        if np.random.random() < aff * 0.35: rows.append({'user_id': user_id, 'author_id': author_id, 'type': 'like', 'ts': ts})
        if np.random.random() < aff * 0.12: rows.append({'user_id': user_id, 'author_id': author_id, 'type': 'comment', 'ts': ts})
        if np.random.random() < (1 - aff) * 0.04: rows.append({'user_id': user_id, 'author_id': author_id, 'type': 'hide', 'ts': ts})
    return pd.DataFrame(rows)

log_df = generate_interaction_log()

def compute_affinity_features(user_id, author_id, log_df, current_time=None):
    """
    Compute all affinity features for a (user, author) pair.
    
    12-year-old version:
    Look at everything this user has done with this author's posts:
    how often do they like? comment? hide? And how long have they been friends?
    
    Staff-level detail:
    - Rates are computed as weighted_reactions / weighted_impressions
    - Time-decay (half-life = 30 days) so recent behavior matters more
    - Returns dict of features ready to be fed into the model
    """
    if current_time is None:
        current_time = datetime(2024, 4, 1)
    
    pair = log_df[(log_df.user_id == user_id) & (log_df.author_id == author_id)].copy()
    
    if len(pair) == 0:
        # Cold start -- no history
        return {'like_rate': 0.0, 'click_rate': 0.0, 'comment_rate': 0.0,
                'hide_rate': 0.0, 'num_past_impressions': 0, 'cold_start': 1}
    
    # Time-decay weights
    days_ago = (current_time - pair['ts']).dt.total_seconds() / 86400
    pair['w'] = np.exp(-0.693 * days_ago / 30)  # half-life = 30 days
    
    w_impressions = pair[pair['type'] == 'impression']['w'].sum()
    if w_impressions < 1e-9:
        return {'like_rate': 0.0, 'click_rate': 0.0, 'comment_rate': 0.0,
                'hide_rate': 0.0, 'num_past_impressions': 0, 'cold_start': 1}
    
    def rate(reaction_type):
        r = pair[pair['type'] == reaction_type]
        return r['w'].sum() / w_impressions if len(r) > 0 else 0.0
    
    return {
        'like_rate':    round(rate('like'),    4),
        'click_rate':   round(rate('click'),   4),
        'comment_rate': round(rate('comment'), 4),
        'hide_rate':    round(rate('hide'),    4),
        'num_past_impressions': int((pair['type'] == 'impression').sum()),
        'cold_start': 0,
    }


# Demonstrate on several (user, author) pairs
test_pairs = [(0, 0), (3, 5), (10, 10), (20, 7), (1, 1)]

print("=== User-Author Affinity Features ===")
print(f"{'Pair':<16} {'true_aff':<11} {'like_rt':<10} {'click_rt':<10} {'cmt_rt':<10} {'hide_rt':<10} {'n_impr'}")
print("-" * 80)
for u, a in test_pairs:
    feats = compute_affinity_features(u, a, log_df)
    ta = true_affinity[u, a]
    print(f"(user={u}, auth={a})  {ta:<11.4f} {feats['like_rate']:<10.4f} "
          f"{feats['click_rate']:<10.4f} {feats['comment_rate']:<10.4f} "
          f"{feats['hide_rate']:<10.4f} {feats['num_past_impressions']}")

In [ ]:
# === Visualizing Affinity Feature Distributions ===

# Compute affinity features for a sample of pairs
sample_pairs = []
for u in range(n_users):
    for a in np.random.choice(n_authors, 3, replace=False):
        feats = compute_affinity_features(u, a, log_df)
        feats.update({'user_id': u, 'author_id': a, 'true_affinity': true_affinity[u, a]})
        sample_pairs.append(feats)

aff_df = pd.DataFrame(sample_pairs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: True affinity vs like_rate scatter
mask = aff_df['cold_start'] == 0
axes[0, 0].scatter(aff_df.loc[mask, 'true_affinity'], aff_df.loc[mask, 'like_rate'],
                   alpha=0.5, color='#2ecc71', s=30)
axes[0, 0].set_xlabel('True Underlying Affinity', fontsize=11)
axes[0, 0].set_ylabel('Observed Like Rate', fontsize=11)
axes[0, 0].set_title('True Affinity vs Like Rate', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Like rate distribution
axes[0, 1].hist(aff_df.loc[mask, 'like_rate'], bins=30, color='#3498db', alpha=0.8, edgecolor='white')
axes[0, 1].set_xlabel('Like Rate', fontsize=11)
axes[0, 1].set_ylabel('Count', fontsize=11)
axes[0, 1].set_title('Like Rate Distribution (Long Tail)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(aff_df.loc[mask, 'like_rate'].median(), color='red',
                   linestyle='--', label=f'Median: {aff_df.loc[mask, "like_rate"].median():.3f}')
axes[0, 1].legend()

# Plot 3: Feature correlation heatmap
feat_cols = ['true_affinity', 'like_rate', 'click_rate', 'comment_rate', 'hide_rate']
corr = aff_df.loc[mask, feat_cols].corr()
im = axes[1, 0].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1, 0].set_xticks(range(len(feat_cols)))
axes[1, 0].set_yticks(range(len(feat_cols)))
axes[1, 0].set_xticklabels([c.replace('_rate', '') for c in feat_cols], rotation=45, ha='right')
axes[1, 0].set_yticklabels([c.replace('_rate', '') for c in feat_cols])
axes[1, 0].set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        axes[1, 0].text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[1, 0])

# Plot 4: Engagement rate by affinity quartile
aff_df.loc[mask, 'affinity_quartile'] = pd.qcut(
    aff_df.loc[mask, 'true_affinity'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
quartile_means = aff_df[mask].groupby('affinity_quartile')[['like_rate', 'comment_rate', 'click_rate']].mean()
quartile_means.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#9b59b6', '#3498db'],
                    edgecolor='white', width=0.7)
axes[1, 1].set_xlabel('Affinity Quartile', fontsize=11)
axes[1, 1].set_ylabel('Reaction Rate', fontsize=11)
axes[1, 1].set_title('Reaction Rates by Affinity Quartile', fontsize=12, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: Users in Q4 (high affinity authors) have dramatically higher")
print("reaction rates. This is WHY affinity is the most predictive feature.")
print("The difference between Q1 and Q4 can be 5-10x in like rate.")

## 4. Training Dataset Construction

### How to Build the Training Examples

Think about what data we collect in production:

```
USER IMPRESSION LOG (what we observe)
=====================================
User 42 saw Post 101 at 2:14pm
User 42 LIKED Post 101 at 2:14pm       <-- explicit positive signal
User 42 saw Post 102 at 2:15pm         <-- no like -> negative for 'like' task
User 42 SHARED Post 103 at 2:16pm      <-- explicit share signal
User 42 scrolled past Post 104 in 0.3s <-- skip signal
```

For each binary task:
- **Positive label**: user performed the reaction on a post they saw
- **Negative label**: user saw the post but did NOT perform the reaction

### Class Imbalance Problem

Reactions are RARE. For every 1 like, there are ~11 impressions with no like. For every 1 share, there are ~50 impressions with no share. If we train on raw data, the model just learns to predict "no reaction" for everything (because that is almost always correct).

**Solutions:**
1. **Negative subsampling**: randomly keep only a fraction of negative examples (e.g., 1:4 ratio)
2. **Class weights in BCE loss**: penalize false negatives more heavily
3. **Focal loss**: automatically down-weights easy negatives

In [ ]:
# === Training Dataset Construction ===

def construct_multitask_training_data(log_df, neg_subsample_ratio=4):
    """
    Build the training dataset for multi-task learning.
    
    12-year-old version:
    For every post a user SAW, we create one row. That row gets labels for
    EVERY task: did they click? like? comment? share? skip?
    If the user never explicitly liked it -> label = 0 for the 'like' task.
    
    Staff-level detail:
    - Start from impressions as the universe (every impression is a training row)
    - Join on reactions to get per-task labels (left join -> NaN = 0)
    - Negative subsampling: keep all positives, sample negatives at 1:neg_ratio
    - This reduces dataset size AND improves model focus on hard examples
    """
    tasks = ['click', 'like', 'comment', 'share']
    
    # Base: all impressions (each is a training example)
    impressions = log_df[log_df['type'] == 'impression'][['user_id', 'author_id']].copy()
    impressions = impressions.drop_duplicates().reset_index(drop=True)
    
    # Add skip signal: mark if user skipped (we simulate this separately)
    impressions['skip_label'] = np.random.binomial(1, 0.35, len(impressions))  # ~35% skip rate
    
    # Add dwell time (regression target)
    # In production: measured from frontend scroll events
    base_dwell = np.random.exponential(4.0, len(impressions))  # avg 4 seconds
    impressions['dwell_time'] = base_dwell
    
    # Add per-task binary labels
    for task in tasks:
        positives = log_df[log_df['type'] == task][['user_id', 'author_id']].drop_duplicates()
        positives[f'label_{task}'] = 1
        impressions = impressions.merge(positives, on=['user_id', 'author_id'], how='left')
        impressions[f'label_{task}'] = impressions[f'label_{task}'].fillna(0).astype(int)
    
    # Print class distribution BEFORE subsampling
    print("=== Class Distribution (raw, before subsampling) ===")
    for task in tasks:
        pos = impressions[f'label_{task}'].sum()
        total = len(impressions)
        print(f"  {task:<10}: {pos:>5} pos / {total-pos:>6} neg  (pos rate: {pos/total*100:.1f}%)")
    
    return impressions


train_data = construct_multitask_training_data(log_df)

# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tasks = ['click', 'like', 'comment', 'share']
pos_rates = [train_data[f'label_{t}'].mean() * 100 for t in tasks]
neg_rates = [100 - r for r in pos_rates]

x = np.arange(len(tasks))
width = 0.35
axes[0].bar(x - width/2, pos_rates, width, label='Positive (reacted)', color='#2ecc71', edgecolor='white')
axes[0].bar(x + width/2, neg_rates, width, label='Negative (saw, no reaction)', color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tasks)
axes[0].set_ylabel('Percentage of Training Set', fontsize=11)
axes[0].set_title('Class Imbalance Per Task', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 110])
for i, (pr, nr) in enumerate(zip(pos_rates, neg_rates)):
    axes[0].text(i - width/2, pr + 1, f'{pr:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Negative subsampling impact on training efficiency
ratios = [1, 2, 4, 10, 20, 50]
dataset_sizes = [len(train_data[train_data['label_like'] == 1]) * (1 + r) for r in ratios]
# AUC roughly saturates after 4:1 ratio (diminishing returns on more negatives)
simulated_auc = [0.62, 0.70, 0.76, 0.78, 0.78, 0.77]

ax2_twin = axes[1].twinx()
axes[1].bar(range(len(ratios)), dataset_sizes, color='#3498db', alpha=0.6, edgecolor='white', label='Dataset size')
ax2_twin.plot(range(len(ratios)), simulated_auc, 'ro-', linewidth=2.5, markersize=8, label='Val AUC (like task)')
axes[1].set_xticks(range(len(ratios)))
axes[1].set_xticklabels([f'1:{r}' for r in ratios])
axes[1].set_xlabel('Positive:Negative Ratio', fontsize=11)
axes[1].set_ylabel('Training Set Size', fontsize=11, color='#3498db')
ax2_twin.set_ylabel('Validation AUC', fontsize=11, color='red')
axes[1].set_title('Negative Subsampling Trade-off', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nTotal training examples: {len(train_data):,}")
print("\nKey insight: 1:4 (positive:negative) ratio gives good AUC without")
print("wasting compute on too many easy negatives.")

## 5. The Loss Function

### Combining Multiple Task Losses

Each task has its own loss function, and we combine them:

```
L_total = α_click  * BCE(P(click), y_click)
        + α_like   * BCE(P(like), y_like)
        + α_comment* BCE(P(comment), y_comment)
        + α_share  * BCE(P(share), y_share)
        + α_skip   * BCE(P(skip), y_skip)
        + α_dwell  * Huber(E[dwell], y_dwell)
```

**Why Huber loss for dwell time?** Dwell time can have huge outliers (someone falls asleep with the feed open -- 3,600 seconds!). Huber loss behaves like MSE for small errors (smooth, nice gradients) and like MAE for large errors (robust to outliers).

**Why higher alpha for rare tasks?** We weight comment and share tasks higher to compensate for their rarer positive signal, forcing the model to pay more attention to them.

In [ ]:
# === Loss Function Visualization and Implementation ===

# --- Visualize Huber vs MSE vs MAE ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

errors = np.linspace(-6, 6, 300)
delta = 1.0  # Huber threshold

mse_loss   = errors**2 / 2
mae_loss   = np.abs(errors)
huber_loss = np.where(np.abs(errors) <= delta,
                      0.5 * errors**2,
                      delta * (np.abs(errors) - 0.5 * delta))

axes[0].plot(errors, mse_loss,   'r-',  linewidth=2.5, label='MSE (blows up on outliers)')
axes[0].plot(errors, mae_loss,   'b-',  linewidth=2.5, label='MAE (gradient discontinuity at 0)')
axes[0].plot(errors, huber_loss, 'g-',  linewidth=2.5, label='Huber (best of both)')
axes[0].axvline(x=-delta, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=+delta, color='gray', linestyle='--', alpha=0.5, label=f'Huber delta = {delta}')
axes[0].set_xlabel('Prediction Error (predicted - actual dwell time)', fontsize=11)
axes[0].set_ylabel('Loss Value', fontsize=11)
axes[0].set_title('Loss Functions for Dwell Time Regression', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 10])

# Plot 2: Multi-task loss weights and their effect
task_names = ['click', 'like', 'comment', 'share', 'skip', 'dwell_time']
alpha_values = [1.0, 2.0, 3.5, 5.0, 1.5, 1.0]  # Alpha weights
colors_bar = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c']

bars = axes[1].bar(task_names, alpha_values, color=colors_bar, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Alpha Weight (α_i)', fontsize=11)
axes[1].set_title('Task Loss Weights in L_total = Σ α_i * L_i', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, alpha in zip(bars, alpha_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{alpha}x', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# --- Implement the multi-task loss ---
class MultiTaskLoss(nn.Module):
    """
    Combined loss for multi-task news feed training.
    
    L_total = sum(alpha_i * L_i)
    Binary tasks: Binary Cross-Entropy
    Regression tasks: Huber (robust to dwell-time outliers)
    """
    def __init__(self, task_weights=None, huber_delta=1.0):
        super().__init__()
        if task_weights is None:
            task_weights = {'click': 1.0, 'like': 2.0, 'comment': 3.5,
                            'share': 5.0, 'skip': 1.5, 'dwell_time': 1.0}
        self.task_weights = task_weights
        self.bce  = nn.BCELoss()
        self.huber = nn.HuberLoss(delta=huber_delta)
    
    def forward(self, predictions, targets, binary_tasks, regression_tasks):
        total_loss = torch.tensor(0.0, requires_grad=True)
        per_task_losses = {}
        
        for task in binary_tasks:
            if task in targets:
                loss = self.bce(predictions[task], targets[task])
                per_task_losses[task] = loss.item()
                total_loss = total_loss + self.task_weights.get(task, 1.0) * loss
        
        for task in regression_tasks:
            if task in targets:
                loss = self.huber(predictions[task], targets[task])
                per_task_losses[task] = loss.item()
                total_loss = total_loss + self.task_weights.get(task, 1.0) * loss
        
        return total_loss, per_task_losses


loss_fn = MultiTaskLoss()
print("=== Multi-Task Loss Function ===")
print(f"Task weights:")
for task, weight in loss_fn.task_weights.items():
    loss_type = 'Huber' if task == 'dwell_time' else 'BCE'
    print(f"  {task:<12}: alpha={weight}  ({loss_type} loss)")
print("\nShare gets alpha=5.0 because it is rare (1 positive per ~50 negatives).")
print("Higher alpha forces the model to allocate more gradient signal to share prediction.")

## 6. Training Loop with Per-Task AUC Tracking

In [ ]:
# === Full Training Loop ===

def generate_synthetic_data(n_samples=5000, n_features=18, seed=42):
    """
    Generate synthetic data that mimics the real feature + label structure.
    Features [0-5]:  affinity features (most important)
    Features [6-10]: user features
    Features [11-17]: post features
    """
    np.random.seed(seed)
    X = np.random.randn(n_samples, n_features)
    
    # Signal is driven mostly by affinity features (first 6 dimensions)
    engagement_signal = (
        0.50 * X[:, 0] +  # like_rate (most predictive)
        0.30 * X[:, 1] +  # click_rate
        0.25 * X[:, 2] +  # comment_rate
        0.20 * X[:, 3] +  # close_friend_flag
        0.10 * X[:, 6] +  # user feature
        0.08 * X[:, 11]   # post feature
    )
    
    noise = lambda: np.random.randn(n_samples) * 0.6
    
    labels = {
        'click':      (engagement_signal + noise() > -0.1).astype(float),
        'like':       (engagement_signal + noise() >  0.5).astype(float),
        'comment':    (engagement_signal + noise() >  1.0).astype(float),
        'share':      (engagement_signal + noise() >  1.5).astype(float),
        'skip':       (-engagement_signal + noise() > 0.8).astype(float),  # anti-correlated
        'dwell_time': np.maximum(0, 4.0 + 3.0 * engagement_signal + noise() * 2.0),
    }
    return X, labels


X, labels = generate_synthetic_data(n_samples=6000, n_features=INPUT_DIM)
split = int(0.8 * len(X))

X_tr, X_te = X[:split], X[split:]
y_tr = {k: v[:split] for k, v in labels.items()}
y_te = {k: v[split:] for k, v in labels.items()}

# Tensors
Xtr_t = torch.FloatTensor(X_tr)
Xte_t = torch.FloatTensor(X_te)
ytr_t = {k: torch.FloatTensor(v) for k, v in y_tr.items()}
yte_t = {k: torch.FloatTensor(v) for k, v in y_te.items()}

# DataLoader
ds = TensorDataset(Xtr_t, *[ytr_t[t] for t in model.all_tasks])
loader = DataLoader(ds, batch_size=256, shuffle=True)

# Re-initialize model
model = MultiTaskNewsFeedDNN(input_dim=INPUT_DIM)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = MultiTaskLoss()

history = {'epoch': [], 'total_loss': [], 'auc_per_task': {t: [] for t in model.binary_tasks}}

N_EPOCHS = 40

print(f"=== Training Multi-Task DNN ({N_EPOCHS} epochs) ===")
print(f"Train: {len(X_tr):,} | Test: {len(X_te):,}")
print(f"Features: {INPUT_DIM} | Batch size: 256")
print()
print(f"{'Epoch':>6}  {'Loss':>8}  " + "  ".join(f"{t:<9}" for t in model.binary_tasks))
print("-" * 70)

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0.0
    
    for batch in loader:
        bx = batch[0]
        by = {t: batch[1 + i] for i, t in enumerate(model.all_tasks)}
        
        optimizer.zero_grad()
        preds = model(bx)
        loss, _ = loss_fn(preds, by, model.binary_tasks, model.regression_tasks)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            te_preds = model(Xte_t)
        
        avg_loss = epoch_loss / len(loader)
        history['epoch'].append(epoch + 1)
        history['total_loss'].append(avg_loss)
        
        auc_vals = []
        for task in model.binary_tasks:
            try:
                auc = roc_auc_score(y_te[task], te_preds[task].numpy())
            except ValueError:
                auc = 0.5
            history['auc_per_task'][task].append(auc)
            auc_vals.append(auc)
        
        aucs_str = "  ".join(f"{v:.3f}    " for v in auc_vals)
        print(f"{epoch+1:>6}  {avg_loss:>8.4f}  {aucs_str}")

print("\nDone.")

In [ ]:
# === Visualize Training Curves ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total loss
axes[0].plot(history['epoch'], history['total_loss'], 'b-o', linewidth=2.5, markersize=7)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Total Multi-Task Loss', fontsize=12)
axes[0].set_title('Training Loss (Σ α_i * L_i)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Per-task AUC curves
palette = {'click': '#3498db', 'like': '#2ecc71', 'comment': '#9b59b6',
           'share': '#e67e22', 'skip': '#e74c3c'}
for task in model.binary_tasks:
    axes[1].plot(history['epoch'], history['auc_per_task'][task], '-o',
                 color=palette[task], linewidth=2.5, markersize=7, label=task)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('ROC-AUC', fontsize=12)
axes[1].set_title('Per-Task AUC on Test Set', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].set_ylim([0.45, 1.05])

plt.tight_layout()
plt.show()

# Final AUC report
model.eval()
with torch.no_grad():
    final_preds = model(Xte_t)

print("=== Final Model Evaluation ===")
print(f"{'Task':<14} {'AUC':<8} {'Pos Rate':<12} {'Notes'}")
print("-" * 60)
for task in model.binary_tasks:
    try:
        auc = roc_auc_score(y_te[task], final_preds[task].numpy())
    except ValueError:
        auc = 0.5
    pos_rate = y_te[task].mean()
    note = 'data-rich, easy' if task == 'click' else ('data-poor, hard' if task == 'share' else '')
    print(f"{task:<14} {auc:<8.4f} {pos_rate:<12.3f} {note}")

dwell_mae = np.mean(np.abs(final_preds['dwell_time'].numpy() - y_te['dwell_time']))
print(f"{'dwell_time':<14} MAE={dwell_mae:.2f}s (regression task)")

## 7. Weighted Engagement Score

Once the model predicts probabilities for each reaction, we combine them into a single **engagement score** used for ranking.

```
Engagement Score = Σ P(reaction_i) * weight_i
```

### Why These Specific Weights?

| Reaction | Weight | Business Reasoning |
|----------|--------|--------------------|
| click | 1 | Weak signal -- someone could misclick or fall for clickbait |
| like | 5 | Deliberate positive signal -- user explicitly approved |
| comment | 10 | User invested effort to write something -- strong engagement |
| share | 20 | User is vouching for this to their friends -- brings new eyeballs |
| friend_request | 30 | Post led to a new connection -- maximum network value |
| dwell_time | 0.1 * seconds | Passive engagement -- capped to avoid sleepers |
| skip | -5 | User scrolled past instantly -- weak negative signal |

In [ ]:
# === Engagement Score Computation and Visualization ===

ENGAGEMENT_WEIGHTS = {
    'click':   1,
    'like':    5,
    'comment': 10,
    'share':   20,
    'skip':    -5,
    # dwell_time handled separately (regression)
}


def compute_engagement_scores(model, feature_matrix, weights, dwell_weight=0.1, dwell_cap=60.0):
    """
    Compute engagement scores for a batch of (user, post) pairs.
    
    12-year-old version:
    For each post, we multiply each predicted probability by its weight
    and sum them all up. The post with the highest total gets shown first.
    
    Staff-level detail:
    - Binary task contributions: P(reaction) * weight  (all in [0, +/-1] * weight)
    - Dwell time contribution:   min(E[dwell], cap) * dwell_weight
    - Capping dwell time prevents outliers (overnight phone sessions) from dominating
    """
    model.eval()
    with torch.no_grad():
        preds = model(torch.FloatTensor(feature_matrix))
    
    scores = np.zeros(len(feature_matrix))
    contributions = {}
    
    for task, w in weights.items():
        if task in preds:
            contrib = preds[task].numpy() * w
            scores += contrib
            contributions[task] = contrib
    
    # Add dwell time contribution (capped)
    dwell_capped = np.minimum(preds['dwell_time'].numpy(), dwell_cap)
    scores += dwell_capped * dwell_weight
    contributions['dwell_time'] = dwell_capped * dwell_weight
    
    return scores, contributions, {t: preds[t].numpy() for t in model.all_tasks}


# Generate 20 candidate posts and score them for a user
np.random.seed(99)
n_candidates = 20
candidate_features = np.random.randn(n_candidates, INPUT_DIM)

scores, contributions, raw_preds = compute_engagement_scores(
    model, candidate_features, ENGAGEMENT_WEIGHTS
)

ranked = np.argsort(-scores)

# Display ranked feed
print("=== Ranked News Feed (Top 10) ===")
print(f"{'Rank':<5} {'Post':<7} {'Score':>8}  {'P(click)':>9} {'P(like)':>8} {'P(cmt)':>8} {'P(share)':>9} {'P(skip)':>8} {'E[dwell]':>9}")
print("-" * 80)
for rank, idx in enumerate(ranked[:10]):
    print(f"{rank+1:<5} Post_{idx:<3} {scores[idx]:>8.3f}  "
          f"{raw_preds['click'][idx]:>9.3f} "
          f"{raw_preds['like'][idx]:>8.3f} "
          f"{raw_preds['comment'][idx]:>8.3f} "
          f"{raw_preds['share'][idx]:>9.3f} "
          f"{raw_preds['skip'][idx]:>8.3f} "
          f"{raw_preds['dwell_time'][idx]:>9.2f}s")

# Visualize score decomposition for top 5 posts
fig, ax = plt.subplots(figsize=(13, 5))

top5 = ranked[:5]
task_order = ['click', 'like', 'comment', 'share', 'skip', 'dwell_time']
colors_stack = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c']

bar_width = 0.6
x = np.arange(5)

bottoms = np.zeros(5)
for task, color in zip(task_order, colors_stack):
    vals = contributions[task][top5]
    ax.bar(x, vals, bar_width, bottom=bottoms, color=color, label=task, edgecolor='white', linewidth=0.5)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([f'Post_{i}\n(rank {r+1})' for r, i in enumerate(top5)])
ax.set_ylabel('Engagement Score Contribution', fontsize=12)
ax.set_title('Engagement Score Decomposition (Top 5 Posts)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nThe share contribution (orange) has a big weight-multiplied impact")
print("even though P(share) is small -- because weight=20.")

## 8. Handling Passive Users

### The Lurker Problem

Many users scroll their feed silently for months -- they never like, comment, or share anything. For these users:
- All binary task probabilities are near zero
- The engagement score for every post is nearly the same (near zero)
- The model cannot distinguish which posts are actually interesting to them

**Solution: Implicit engagement signals**

Even passive users unconsciously signal preferences through:
1. **Dwell time**: They spend 15 seconds reading a sports post vs 0.3 seconds on a political post
2. **Skip**: They immediately scroll past certain content types

In [ ]:
# === Passive Users: Dwell Time as the Primary Signal ===

np.random.seed(42)
n_posts_per_user = 50

# Simulate three types of users
def simulate_user_sessions(n_posts=50, user_type='active'):
    topics = np.random.choice(['sports', 'food', 'tech', 'music', 'politics'], n_posts)
    
    if user_type == 'active':
        # Active users: like 15%, comment 5%, share 2%
        likes    = np.random.binomial(1, 0.15, n_posts)
        comments = np.random.binomial(1, 0.05, n_posts)
        shares   = np.random.binomial(1, 0.02, n_posts)
        dwell    = np.random.exponential(8, n_posts)
    elif user_type == 'passive':
        # Passive users: almost no explicit reactions, BUT dwell varies by topic
        likes    = np.random.binomial(1, 0.005, n_posts)
        comments = np.random.binomial(1, 0.001, n_posts)
        shares   = np.zeros(n_posts, dtype=int)
        # Dwell time varies by topic (reveals preferences!)
        dwell = np.array([np.random.exponential(12 if t == 'sports' else 2) for t in topics])
    else:  # ghost user -- barely any signal
        likes    = np.zeros(n_posts, dtype=int)
        comments = np.zeros(n_posts, dtype=int)
        shares   = np.zeros(n_posts, dtype=int)
        dwell    = np.random.exponential(1.5, n_posts)
    
    skips = (dwell < 0.5).astype(int)
    return pd.DataFrame({'topic': topics, 'dwell': dwell, 'like': likes,
                         'comment': comments, 'share': shares, 'skip': skips})


active_df  = simulate_user_sessions(user_type='active')
passive_df = simulate_user_sessions(user_type='passive')
ghost_df   = simulate_user_sessions(user_type='ghost')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_user_signals(ax, df, title, color):
    # Bar: explicit reaction rates
    reaction_rates_plot = {
        'like': df['like'].mean(),
        'comment': df['comment'].mean(),
        'share': df['share'].mean(),
    }
    ax2 = ax.twinx()
    bars = ax.bar(reaction_rates_plot.keys(), reaction_rates_plot.values(),
                  color=color, alpha=0.7, edgecolor='white', label='Reaction rate')
    # Overlay: dwell time by topic (the real signal for passive users)
    topic_dwell = df.groupby('topic')['dwell'].mean().sort_values(ascending=False)
    ax2.bar([f"dwell:\n{t[:4]}" for t in topic_dwell.index], topic_dwell.values,
            color='#95a5a6', alpha=0.5, edgecolor='white', label='Dwell (s)')
    ax.set_ylabel('Reaction Rate', fontsize=10, color=color)
    ax2.set_ylabel('Avg Dwell Time (s)', fontsize=10, color='#555')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=30)

plot_user_signals(axes[0], active_df,  'Active User (Likes, Comments)', '#2ecc71')
plot_user_signals(axes[1], passive_df, 'Passive User (Lurker)',         '#e67e22')
plot_user_signals(axes[2], ghost_df,   'Ghost User (Barely Online)',     '#e74c3c')

plt.tight_layout()
plt.show()

print("=== Why Dwell Time Matters for Passive Users ===")
print(f"\nActive user  | explicit signals: like={active_df['like'].mean():.1%}, comment={active_df['comment'].mean():.1%}")
print(f"Passive user | explicit signals: like={passive_df['like'].mean():.1%}, comment={passive_df['comment'].mean():.1%}")
print(f"\nPassive user dwell by topic:")
for topic, dwell in passive_df.groupby('topic')['dwell'].mean().sort_values(ascending=False).items():
    print(f"  {topic:<12}: {dwell:.1f}s avg dwell  ", end='')
    print("<-- LOVES this topic (revealed by dwell!)" if dwell > 6 else "")
print("\nDwell time reveals the passive user's topic preferences -- something that")
print("likes/comments would completely miss.")

## 9. Positional Bias

### The Problem

Imagine you are at a buffet. The first dish you see at eye level is a random pasta salad. You take some, even though you might have preferred the salmon at the end of the table -- but you never got that far.

**Positional bias** is the same phenomenon in news feeds: users are more likely to click, like, or interact with posts that appear at the **top** of their feed -- not because those posts are better, but simply because they are first.

This creates a bias in training data:
- Posts shown at position 1 get many clicks --> model learns they are "good"
- But they got clicks because of position, not quality
- Model reinforces bad ranking: position 1 gets more clicks, which trains the model to rank them higher, which gives them even more clicks...

### Solution: Inverse Propensity Weighting (IPW)

We estimate the probability that a click was due to position (the "propensity"), then **down-weight** training examples from high-position posts accordingly.

In [ ]:
# === Positional Bias: Measurement and Correction ===

# --- Simulate positional bias data ---
np.random.seed(42)
n_sessions = 5000

# Position propensity: probability of clicking purely due to position
# (independent of post quality)
positions = np.arange(1, 21)  # feed positions 1-20
true_propensity = 0.35 * np.exp(-0.25 * (positions - 1))  # decays with position

# Generate observed click data
observed_click_rates = []
for pos in positions:
    # Observed click = quality effect + position effect
    post_quality = np.random.uniform(0.05, 0.25, n_sessions // len(positions))
    position_boost = true_propensity[pos - 1]
    actual_rate = np.clip(post_quality + position_boost, 0, 1)
    clicks = np.random.binomial(1, actual_rate)
    observed_click_rates.append(clicks.mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Observed click rate by position
axes[0].bar(positions, observed_click_rates, color='#e74c3c', alpha=0.8, edgecolor='white', label='Observed CTR')
axes[0].plot(positions, true_propensity, 'b-o', linewidth=2.5, markersize=6, label='True positional propensity')
axes[0].set_xlabel('Feed Position', fontsize=12)
axes[0].set_ylabel('Click Rate', fontsize=12)
axes[0].set_title('Positional Bias: CTR Drops Sharply with Position', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(positions[::2])

# Plot 2: IPW correction
# IPW weight for each click = 1 / P(observed | position)
ipw_weights = 1.0 / (true_propensity + 1e-9)
# Normalize weights
ipw_weights = ipw_weights / ipw_weights.mean()

# Without IPW: model is fooled by position
naive_quality_estimate = np.array(observed_click_rates)
# With IPW: correct for position
ipw_quality_estimate = np.array(observed_click_rates) * (1.0 / (true_propensity / true_propensity.mean() + 1e-9))
ipw_quality_estimate = np.clip(ipw_quality_estimate, 0, 1)

axes[1].plot(positions, naive_quality_estimate, 'r-o', linewidth=2.5, markersize=6, label='Naive (biased by position)')
axes[1].plot(positions, ipw_quality_estimate, 'g-o', linewidth=2.5, markersize=6, label='IPW-corrected')
axes[1].axhline(y=0.15, color='gray', linestyle='--', alpha=0.7, label='True avg quality (≈0.15)')
axes[1].set_xlabel('Feed Position', fontsize=12)
axes[1].set_ylabel('Estimated Post Quality', fontsize=12)
axes[1].set_title('Inverse Propensity Weighting Correction', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(positions[::2])

plt.tight_layout()
plt.show()


def compute_ipw_weights(positions_batch, propensity_model):
    """
    Compute IPW sample weights for training.
    
    12-year-old version:
    A click from position 1 is less informative (maybe they clicked just
    because it was first). We give it a smaller weight in training.
    A click from position 15 is very informative (they scrolled that far
    and STILL clicked). We give it a bigger weight.
    
    Staff-level detail:
    IPW weight_i = 1 / P(click | position_i)
    Weighted BCE loss = -sum( weight_i * [y_i * log(p_i) + (1-y_i) * log(1-p_i)] )
    """
    propensities = propensity_model(positions_batch)
    weights = 1.0 / (propensities + 1e-8)
    weights = weights / weights.mean()  # normalize so average weight = 1
    return weights


print("=== IPW Summary ===")
print("\nPositional propensities and corresponding IPW weights:")
print(f"{'Position':<10} {'P(click|pos)':<16} {'IPW weight':<12} {'Interpretation'}")
print("-" * 65)
for pos in [1, 3, 5, 10, 15, 20]:
    prop = true_propensity[pos-1]
    weight = 1.0 / (prop + 1e-9)
    note = 'down-weight (position effect strong)' if pos <= 3 else ('up-weight (position effect weak)' if pos >= 15 else '')
    print(f"{pos:<10} {prop:<16.4f} {weight:<12.2f} {note}")

## 10. Cold-Start for New Users

### The Problem

A brand-new user has:
- No interaction history
- No affinity features (the most predictive signal!)
- No past reactions to learn from

This is called the **cold-start problem**. It is like trying to recommend movies to someone who just walked into a video store and has never seen a movie before.

### Fallback Strategy: Content-Based Ranking

When affinity features are unavailable, fall back to:
1. **Demographics-based**: Use age, location, device type to guess likely interests
2. **Popularity-based**: Show trending posts in their region
3. **Explicit onboarding**: Ask new users to select 3-5 topics they like
4. **Exploration**: Show a diverse mix and quickly learn from early clicks

In [ ]:
# === Cold-Start: Content-Based Fallback ===

import warnings
warnings.filterwarnings('ignore')


class ColdStartFallback:
    """
    Content-based ranking for new users with no interaction history.
    
    12-year-old version:
    When we know nothing about a user, we use the few clues we have:
    - Are they 16 or 45? (different interests)
    - Are they in Brazil or Japan? (different trending content)
    - What topics did they pick during signup?
    We match those clues against post content to make our best guess.
    
    Staff-level detail:
    - Phase 1 (first week): fallback to demographic + popularity signals
    - Phase 2 (after 10 interactions): partial affinity features become available
    - Phase 3 (after 30 interactions): full personalization kicks in
    """
    
    def __init__(self):
        # Topic engagement rates by age group (learned from warm users)
        self.topic_rates_by_age = {
            'teen':   {'music': 0.35, 'sports': 0.25, 'memes': 0.45, 'travel': 0.10, 'tech': 0.15},
            'young_adult': {'music': 0.25, 'sports': 0.30, 'memes': 0.25, 'travel': 0.25, 'tech': 0.25},
            'adult':  {'sports': 0.30, 'travel': 0.30, 'tech': 0.25, 'news': 0.35, 'food': 0.30},
            'senior': {'news': 0.40, 'family': 0.45, 'travel': 0.25, 'food': 0.30, 'sports': 0.20},
        }
    
    def get_age_group(self, age):
        if age < 20:   return 'teen'
        elif age < 35: return 'young_adult'
        elif age < 55: return 'adult'
        else:          return 'senior'
    
    def rank_for_new_user(self, user_profile, candidate_posts, n_interactions=0):
        """
        Rank posts for a user with limited history.
        As n_interactions grows, personalization increases.
        """
        age_group = self.get_age_group(user_profile['age'])
        topic_prefs = self.topic_rates_by_age.get(age_group, {})
        
        scores = []
        for post in candidate_posts:
            # Content-based score from demographic match
            content_score = topic_prefs.get(post['topic'], 0.10)  # 0.10 default
            # Popularity score
            popularity_score = post['likes'] / 1000.0  # normalized
            # Exploration bonus (small random boost for diversity)
            explore_bonus = np.random.uniform(0, 0.05)
            
            # Weight: as user gets more interactions, rely less on demographics
            cold_weight = max(0, 1.0 - n_interactions / 30.0)
            
            final_score = (cold_weight * (0.6 * content_score + 0.3 * popularity_score + 0.1 * explore_bonus) +
                          (1 - cold_weight) * np.random.uniform(0.2, 0.8))  # placeholder for real model
            scores.append(final_score)
        
        ranked = np.argsort(-np.array(scores))
        return ranked, scores


# Simulate cold-start for different user profiles
fallback = ColdStartFallback()

np.random.seed(7)
topics_pool = ['sports', 'music', 'memes', 'tech', 'travel', 'news', 'food', 'family']
n_posts = 20
candidate_posts = [
    {'post_id': i, 'topic': topics_pool[i % len(topics_pool)],
     'likes': np.random.randint(10, 2000)}
    for i in range(n_posts)
]

users = [
    {'name': 'Alex (16, teen)',     'age': 16},
    {'name': 'Sam (28, young_adult)', 'age': 28},
    {'name': 'Jordan (47, adult)',  'age': 47},
    {'name': 'Pat (65, senior)',    'age': 65},
]

print("=== Cold-Start: Top 5 Posts for Different User Profiles ===")
print("(No interaction history -- ranking purely from demographics + popularity)\n")

for user in users:
    ranked, scores = fallback.rank_for_new_user(user, candidate_posts, n_interactions=0)
    print(f"User: {user['name']} | Age group: {fallback.get_age_group(user['age'])}")
    for rank, idx in enumerate(ranked[:5]):
        p = candidate_posts[idx]
        print(f"  #{rank+1}: Post_{p['post_id']:<3} topic={p['topic']:<10} likes={p['likes']:<6} score={scores[idx]:.3f}")
    print()

# Visualize: how fast does personalization take over?
interaction_counts = np.arange(0, 31)
cold_weights = np.maximum(0, 1.0 - interaction_counts / 30.0)
warm_weights = 1.0 - cold_weights

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(interaction_counts, cold_weights, alpha=0.7, color='#e74c3c', label='Demographic fallback weight')
ax.fill_between(interaction_counts, warm_weights, alpha=0.7, color='#2ecc71', label='Personalization model weight')
ax.set_xlabel('Number of User Interactions (clicks, likes, etc.)', fontsize=12)
ax.set_ylabel('Weight in Final Score', fontsize=12)
ax.set_title('Cold-Start: Transition from Demographics to Personalization', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Multi-task learning beats N independent models** because sparse reactions (shares, comments) get a free ride from the shared backbone trained on abundant data (clicks)

2. **User-author affinity is the single most predictive feature** (per Facebook research) -- like_rate, comment_rate, close_friend_flag are your highest-value inputs

3. **The engagement score formula** `Σ P(reaction_i) * weight_i` allows the business to tune what matters: shares (w=20) are worth much more than clicks (w=1)

4. **Passive users need implicit signals**: add dwell_time regression head and skip binary head to capture engagement for users who never explicitly react

5. **Class imbalance is real**: shares are 50x rarer than clicks -- use higher alpha weights for rare tasks and consider negative subsampling

6. **Positional bias corrupts training data**: posts at position 1 get clicks because of position, not quality -- use inverse propensity weighting to correct for this

7. **Cold-start fallback**: new users get demographic + popularity-based ranking until they accumulate ~30 interactions to enable full personalization

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)